In [1]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,\
    ha,SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,hi,E_fcis,\
        NESFermionHopRule,compute_qgt,sampler_info
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time

# ========== 你原有全局参数（直接复用） ==========
# 单系统希尔伯特空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
K = 3  # NES 扩展副本数
hi_ext = hi ** K  # 扩展希尔伯特空间
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
single_edges = ((0, 1), (2, 3))  # 费米子跃迁边
g = nk.graph.Graph(edges=single_edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)

total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: With many Markov Chains (e.g GPUs), n_discard_per_chain>5 is often inefficient.

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


In [ ]:
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER =400
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
Natural_Grad = True


total_ansatz = NESTotalAnsatz(4,K,20,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
    

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)  # 转为jax数组（关键修复）

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=16,
    sweep_size=20
)


# 采样器状态初始化（替代原 init_sampler_state）
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# ==================== 训练循环（仅替换采样部分） ====================
print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (NetKet 自定义采样器 + 朴素梯度下降)")
print("="*60)
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")

history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_2st': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[],
    'log_Psi_mean':[],
    'log_Psi_min':[],
    'log_Psi_max':[],
    'grad_norm':[],
}

start_time = time.time()
for step in range(N_ITER):
    # 2. 正式采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine, parameters=total_params, 
        state=sampler_state, chain_length=N_SAMPLES_PER_CHAIN
    )
        # 3. 维度重塑，适配梯度函数输入
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, 4)
    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_matrix_machine=total_matrix_machine,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,4))
    #grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    if Natural_Grad == True:
        qgt_reg, unravel_fn = compute_qgt(total_machine, total_params, samples.reshape(-1,K,4), diag_shift=0.1)
        # # 自然梯度求解
        natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
        natural_grad = grad_unravel_fn(natural_grad_flat)
        grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
    eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
    grad_norm = jnp.linalg.norm(grad_flat)
    
    
    history['step'].append(step)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['loss'].append(loss_mean)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['grad_norm'].append(grad_norm)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['energy_2st'].append(eig_vals[2])
    history['params'].append(total_params)
    # 5. 记录历史
    if step % 50 == 0 or step == N_ITER - 1:
        # --------------------- 【NES-VMC 监控模板】直接用 ---------------------
        # 1. 监控 log_Psi
        #log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
        print(f"log_Psi: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")

        # 2. 监控梯度范数
        
        print(f"grad norm = {grad_norm:.4f}")
        print(f"Step {step:3d} | Loss: {loss_mean}|0st能量={eig_vals[0]:.8f} Ha｜1st能量={eig_vals[1]:.8f} Ha｜2st能量={eig_vals[2]:.8f} Ha")
        # print(f'grad={grad_flat[30:31]}')
        print('#-----------------------------------------#')


end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
print("\n" + "="*60)
print(f"训练完成!")
print("="*60)

In [ ]:
Natural_Grad = True

$$
\begin{align*}
\Psi(\mathbf{x})^{-1}\hat{\mathcal{H}}\Psi(\mathbf{x})
&= \mathrm{Tr}\left[ \Psi^{-1}(\mathbf{x})\hat{H}\Psi(\mathbf{x}) \right]
\end{align*}
$$

In [ ]:
import pickle
import os  # 加上这个
# 自动创建 data 文件夹（关键修复）
os.makedirs('./data', exist_ok=True)
if Natural_Grad == True:
    print('保存自然梯度历史记录')
    # 保存 history
    with open('./data/history_natural_gradient_K3.pkl', 'wb') as f:
        pickle.dump(history, f)
else:
    # 保存 history
    with open('./data/history_plain_gradient_K3.pkl', 'wb') as f:
        pickle.dump(history, f)

print("保存成功！")

In [ ]:
import pickle
history_natural= pickle.load(open('./data/history_natural_gradient_K3.pkl', 'rb'))
history_plain= pickle.load(open('./data/history_plain_gradient_K3.pkl', 'rb'))


In [ ]:
import matplotlib.pyplot as plt
import sys
import numpy as np
sys.path.append('..')
from NES_VMC import E_fcis

# 创建 3行3列 子图
fig, axs = plt.subplots(3, 3, figsize=(12, 9))
fig.suptitle('Natural Excited State-VMC for $H_2$ K=3 ', fontsize=14)

# -------------------- 第一行：各能级能量演化 --------------------
# 0态能量
axs[0,0].plot(history_natural['energy_0st'], color='orange', label='natural gradient')
axs[0,0].plot(history_plain['energy_0st'], color='blue', label='plain gradient')
axs[0,0].hlines(E_fcis[0], 0, len(history_natural['energy_0st']), linestyle='--', color='red', label='FCI')
axs[0,0].set_title('0st Energy')
axs[0,0].set_ylabel('energy')
axs[0,0].set_xlabel('step')
axs[0,0].legend()

# 1态能量
axs[0,1].plot(history_natural['energy_1st'], color='orange', label='natural gradient')
axs[0,1].plot(history_plain['energy_1st'], color='blue', label='plain gradient')
axs[0,1].hlines(E_fcis[1], 0, len(history_natural['energy_1st']), linestyle='--', color='red')
axs[0,1].set_title('1st Energy')
axs[0,1].set_ylabel('energy')
axs[0,1].set_xlabel('step')
axs[0,1].legend()

# 2态能量
axs[0,2].plot(history_natural['energy_2st'], color='orange', label='natural gradient')
axs[0,2].plot(history_plain['energy_2st'], color='blue', label='plain gradient')
axs[0,2].hlines(E_fcis[2], 0, len(history_natural['energy_2st']), linestyle='--', color='red')
axs[0,2].set_title('2st Energy')
axs[0,2].set_ylabel('energy')
axs[0,2].set_xlabel('step')
axs[0,2].legend()

# -------------------- 第二行：各能级能量误差演化 --------------------
err0 = np.array(history_natural['energy_0st']) - E_fcis[0]
err1 = np.array(history_natural['energy_1st']) - E_fcis[1]
err2 = np.array(history_natural['energy_2st']) - E_fcis[2]

axs[1,0].plot(err0, color='orange', label='natural gradient')
axs[1,0].set_title('0st Energy Error')
axs[1,0].set_xlabel('step')
axs[1,0].set_ylabel('energy error')
axs[1,0].legend()

axs[1,1].plot(err1, color='orange', label='natural gradient')
axs[1,1].set_title('1st Energy Error')
axs[1,1].set_xlabel('step')
axs[1,1].set_ylabel('energy error')
axs[1,1].legend()

axs[1,2].plot(err2, color='orange', label='natural gradient')
axs[1,2].set_title('2st Energy Error')
axs[1,2].set_xlabel('step')
axs[1,2].set_ylabel('energy error')
axs[1,2].legend()

# -------------------- 第三行前两个：损失 & 梯度范数 --------------------
axs[2,0].plot(history_natural['loss'], color='orange', label='natural gradient')
axs[2,0].plot(history_plain['loss'], color='blue', label='plain gradient')
axs[2,0].set_title('loss')
axs[2,0].set_xlabel('step')
axs[2,0].set_ylabel('loss')
axs[2,0].legend()

axs[2,1].plot(history_natural['grad_norm'], color='orange', label='natural gradient')
axs[2,1].plot(history_plain['grad_norm'], color='blue', label='plain gradient')
axs[2,1].set_title('grad_norm')
axs[2,1].set_xlabel('step')
axs[2,1].set_ylabel('grad_norm')
axs[2,1].legend()

# -------------------- 第三行第三个：柱状图 - 最终能级误差 --------------------
# 取最后一步的误差
final_err = [err0[-1], err1[-1], err2[-1]]
level_labels = ['0st', '1st', '2nd']
x = np.arange(len(level_labels))
width = 0.4

# 绘制柱状图
axs[2,2].bar(x, final_err, width, color='orange', label='Final Error')
axs[2,2].axhline(y=0, color='red', linestyle='--')  # 零误差参考线
axs[2,2].set_title('Final Energy Error per Level')
axs[2,2].set_xticks(x)
axs[2,2].set_xticklabels(level_labels)
axs[2,2].set_ylabel('final energy error (Ha)')
axs[2,2].legend()

# 自动调整布局，防止标题/标签重叠
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

def sampler_info(samples:jnp.array,K:int):
    test_samples = np.array(samples.reshape(-1, 4*K))
    count = Counter(tuple(each_row.tolist()) for each_row in test_samples)
    for tpl, count_ in count.items():
        print(f"元组 {tpl} 出现了 {count_} 次")
    return count

K = 3
sampler_info(history_natural['samples'][100],K)

In [ ]:
history_natural['grad_norm'][180]
abnormal_samples = history_natural['samples'][180]

In [ ]:
abnormal_samples = history_natural['samples'][180]
output = total_machine(history_natural['params'][179],abnormal_samples)
print(output.max())

In [ ]:

fig,axs = plt.subplots(1,2,figsize=(12,10))
axs[0].plot(range(150,200),history_natural['grad_norm'][150:200])
axs[1].plot(range(150,200),history_natural['loss'][150:200])

In [ ]:
from NES_VMC import NES_loss_energy
K=2
hi_ext = hi**K
total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
    
loss_batch,E_L_batch  = NES_loss_energy(ha=ha,total_matrix_machine=total_matrix_machine,
                single_machine_list=single_machine_list,
                total_params=total_params,
                x=abnormal_samples.reshape(-1,K,4))

M 为矩阵： $\ln{\det{M}} = Tr(\ln{M})$


In [ ]:
def Ham_psi(ha: nk.operator.DiscreteOperator, single_machine, params, x):
    """
    🔥 同时支持：
    - 单个态 x: (n_spin,)
    - 批量态 x: (batch_size, n_spin)
    """
    # ======================
    # 核心：自动给单个样本增加 batch 维度
    # ======================
    is_single = (x.ndim == 1)
    if is_single:
        x = x[None, :]  # (n_spin,) → (1, n_spin)

    # ======================
    # 向量化计算（批处理）
    # ======================
    def _single_hpsi(x_single):
        x_primes, mels = ha.get_conn_padded(x_single)
        log_psi_vals = single_machine(params, x_primes)
        psi_vals = jnp.exp(log_psi_vals)
        return jnp.sum(mels * psi_vals)

    # 批量处理
    H_psi_batch = jax.vmap(_single_hpsi)(x)

    # ======================
    # 如果是单个输入，就压回单个输出
    # ======================
    if is_single:
        return H_psi_batch[0]
    else:
        return H_psi_batch
    
def Ham_Psi(ha, single_machine_list, total_params, x):
    K = len(single_machine_list)
    # ======================
    # 核心：单样本 与 批处理 自动兼容
    # ======================
    if x.ndim == 2:
        # 输入形状：(K, n_spin) → 单个扩展态 → 返回 (K,K)
        def _single_HamPsi(x_single):
            HPsi = jnp.zeros((K, K), dtype=complex)
            for i in range(K):
                xi = x_single[i]  # 单态：(4,)
                for j in range(K):
                    machine_j = single_machine_list[j]
                    params_j = total_params['single_ansatz_list'][j]
                    val = Ham_psi(ha, machine_j, params_j, xi)
                    HPsi = HPsi.at[i, j].set(val)
            return HPsi
        
        return _single_HamPsi(x)

    elif x.ndim == 3:
        # 输入形状：(batch, K, n_spin) → 批量 → 返回 (batch, K, K)
        def _single_HamPsi(x_single):
            HPsi = jnp.zeros((K, K), dtype=complex)
            for i in range(K):
                xi = x_single[i]
                for j in range(K):
                    machine_j = single_machine_list[j]
                    params_j = total_params['single_ansatz_list'][j]
                    val = Ham_psi(ha, machine_j, params_j, xi)
                    HPsi = HPsi.at[i, j].set(val)
            return HPsi
        
        # 自动批处理！
        return jax.vmap(_single_HamPsi)(x)

    else:
        raise ValueError(f"不支持的输入形状: {x.shape}")


In [ ]:
hi_ext.all_states()[1]

In [ ]:
test_samples = hi_ext.all_states()[1].reshape(K,4)
single_machine_list[0](total_params['single_ansatz_list'][0], test_samples[0])
total_machine(total_params, test_samples)

In [ ]:
M = total_matrix_machine(total_params, test_samples)

In [ ]:
jnp.linalg.det(M)